# Batch 5A — Control-Gate Width Sensitivity Sweep

This notebook runs the registered `rq2_control_gate_width_v1` study over fixed top-1, top-2, top-3, and top-5 control gates. It changes only the gate width, preserves the 24-record evidence-window cap, and uses the same pinned 4-bit Qwen2.5-7B selector as RQ2 v3. The released adaptive RQ2 v3 result is loaded only as an unchanged reference.

Before selecting **Runtime → Run all**:

1. Select **Runtime → Change runtime type → A100 GPU**. If A100 is unavailable, select **T4 GPU** (L4 is also suitable).
2. Approve the Google Drive mount. The runner checkpoints after every question and safely resumes from the same Drive folder after a disconnect.
3. Keep the registered output directory name unchanged: `MyDrive/rq2_control_gate_width_v1`. Do not reuse that directory for another model, code version, or experimental design.

The notebook checks out the immutable Batch 5A runner commit, verifies the frozen RQ2 v3 archive, prepares and audits all 544 fixed-width contexts, verifies the pinned Hugging Face model revision, runs the full GPU sweep, and validates all eight completed contract files plus the summary. No GitHub or Hugging Face credential is required because the repository and model are public.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
OUTPUT_DIR = Path('/content/drive/MyDrive/rq2_control_gate_width_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent checkpoint directory:', OUTPUT_DIR)


In [ ]:
import torch

assert torch.cuda.is_available(), (
    'CUDA GPU not found. Choose Runtime > Change runtime type > A100 GPU '
    'or T4 GPU, then rerun this notebook.'
)
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU: {gpu_name} ({gpu_memory_gb:.1f} GiB)')
if not any(label in gpu_name.upper() for label in ('A100', 'T4', 'L4')):
    print('Note: this GPU is not A100/T4/L4, but the registered 4-bit run may still work.')


In [ ]:
import shutil
import subprocess
from pathlib import Path

REPOSITORY_URL = 'https://github.com/Wenjia1215/ComplianceGPT.git'
SOURCE_COMMIT = '372632f81ed4ceeb18021690d03ad0874d988092'
REPO_DIR = Path('/content/ComplianceGPT')

# /content is disposable; Drive checkpoints are never removed here.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
REPO_DIR.mkdir(parents=True)
subprocess.run(['git', 'init', '-q'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'remote', 'add', 'origin', REPOSITORY_URL], cwd=REPO_DIR, check=True)
subprocess.run(
    ['git', 'fetch', '-q', '--depth', '1', 'origin', SOURCE_COMMIT],
    cwd=REPO_DIR, check=True,
)
subprocess.run(['git', 'checkout', '-q', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
checked_out_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
assert checked_out_commit == SOURCE_COMMIT, (checked_out_commit, SOURCE_COMMIT)
runner = REPO_DIR / 'experiments/answerer_comparison/rq2_control_gate_width/run_control_gate_width.py'
assert runner.is_file(), f'Batch 5A runner is missing: {runner}'
print('Batch 5A source commit:', checked_out_commit)
print('Runner:', runner)


In [ ]:
import importlib.metadata
import subprocess
import sys

requirements = REPO_DIR / 'experiments/answerer_comparison/rq2_matched/requirements-colab.txt'
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', '-r', str(requirements)],
    check=True,
)
for package in ('transformers', 'accelerate', 'bitsandbytes', 'huggingface-hub'):
    print(f'{package}: {importlib.metadata.version(package)}')


In [ ]:
import csv
import hashlib
import json
from IPython.display import Markdown, display

FROZEN_ARCHIVE_SHA256 = '56a70db6eca0420df6affbe63c859418ac423b54d80d3d1eae7bf785f0f63328'
frozen_archive = REPO_DIR / 'experiments/answerer_comparison/rq2_matched/results_v3/compliancegpt_rq2_matched_v3.zip'
actual_archive_sha256 = hashlib.sha256(frozen_archive.read_bytes()).hexdigest()
assert actual_archive_sha256 == FROZEN_ARCHIVE_SHA256, (
    actual_archive_sha256, FROZEN_ARCHIVE_SHA256
)
print('Frozen RQ2 v3 archive verified:', actual_archive_sha256)

prepare_command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--prepare-only',
]
subprocess.run(prepare_command, cwd=REPO_DIR, check=True)
diagnostics_path = OUTPUT_DIR / 'preflight/window_diagnostics.csv'
with diagnostics_path.open('r', encoding='utf-8', newline='') as handle:
    diagnostic_rows = list(csv.DictReader(handle))
fixed_width_rows = sum(row.get('configuration', '').startswith('top') for row in diagnostic_rows)
adaptive_reference_rows = sum(row.get('configuration') == 'adaptive_v3' for row in diagnostic_rows)
assert fixed_width_rows == 544, f'Expected 544 fixed-width contexts, found {fixed_width_rows}'
assert adaptive_reference_rows == 136, (
    f'Expected 136 adaptive-reference rows, found {adaptive_reference_rows}'
)
preflight_summary = json.loads((OUTPUT_DIR / 'preflight/summary.json').read_text(encoding='utf-8'))
assert preflight_summary.get('scope') == 'gold-aware evidence-window opportunity audit; no selector inference'
print(
    'Prepared-context audit passed: '
    f'{fixed_width_rows} fixed-width contexts + '
    f'{adaptive_reference_rows} frozen adaptive-reference rows.'
)
display(Markdown((OUTPUT_DIR / 'PREPARED_CONTEXTS.md').read_text(encoding='utf-8')))


In [ ]:
from huggingface_hub import HfApi

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
MODEL_REVISION = 'a09a35458c702b33eeacc393d103063234e8bc28'
model_info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION)
assert model_info.sha == MODEL_REVISION, (model_info.sha, MODEL_REVISION)
assert torch.cuda.is_available(), 'CUDA was lost before model loading.'
print(f'Pinned model verified: {MODEL_ID}@{model_info.sha}')
print('GPU ready:', torch.cuda.get_device_name(0))


In [ ]:
import os
from collections import deque

run_command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
]
run_environment = os.environ.copy()
run_environment.setdefault('TOKENIZERS_PARALLELISM', 'false')
run_environment.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')
tail = deque(maxlen=400)
print('Starting the full top-1/2/3/5 selector sweep.')
print('If the runtime disconnects, reconnect and run this notebook again; completed rows are validated and skipped.')
process = subprocess.Popen(
    run_command, cwd=REPO_DIR, env=run_environment,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
    tail.append(line)
return_code = process.wait()
failure_log = OUTPUT_DIR / 'runner_failure_tail.log'
if return_code != 0:
    failure_log.write_text(''.join(tail), encoding='utf-8')
    print(f'Batch 5A failed. The final runner output is saved at {failure_log}.')
    raise subprocess.CalledProcessError(return_code, run_command)
if failure_log.exists():
    failure_log.unlink()
print('Batch 5A runner completed successfully.')


In [ ]:
expected_contract_rows = {
    **{f'top{width}_rev5_compliancegpt.csv': 100 for width in (1, 2, 3, 5)},
    **{f'top{width}_rev4_compliancegpt.csv': 36 for width in (1, 2, 3, 5)},
}
contracts_dir = OUTPUT_DIR / 'contracts'
problems = []
for filename, expected in expected_contract_rows.items():
    path = contracts_dir / filename
    if not path.is_file():
        problems.append(f'{filename}: missing; expected {expected} rows')
        continue
    with path.open('r', encoding='utf-8', newline='') as handle:
        row_count = sum(1 for _ in csv.DictReader(handle))
    if row_count != expected:
        problems.append(f'{filename}: {row_count} rows; expected {expected}')
    else:
        print(f'{filename}: {row_count} rows')

summary_path = OUTPUT_DIR / 'summary.json'
if not summary_path.is_file():
    problems.append('summary.json is missing')
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    if summary.get('result_id') != 'rq2_control_gate_width_v1':
        problems.append(f"Unexpected result identity: {summary.get('result_id')}")
    expected_configurations = {
        f'{revision}_{gate}'
        for revision in ('rev5', 'rev4')
        for gate in ('adaptive_v3', 'top1', 'top2', 'top3', 'top5')
    }
    observed_configurations = set(summary.get('configurations', {}))
    missing_configurations = sorted(expected_configurations - observed_configurations)
    if missing_configurations:
        problems.append('Missing summary configurations: ' + ', '.join(missing_configurations))

if problems:
    print('\n'.join(problems))
    raise RuntimeError(
        'The Batch 5A run is incomplete. Rerun the full-run cell; valid completed rows will be skipped.'
    )
result_zip = OUTPUT_DIR.parent / f'{OUTPUT_DIR.name}.zip'
assert result_zip.is_file(), f'Result archive is missing: {result_zip}'
result_zip_sha256 = hashlib.sha256(result_zip.read_bytes()).hexdigest()
print('Validated result archive:', result_zip)
print('Result archive SHA-256:', result_zip_sha256)
display(Markdown((OUTPUT_DIR / 'SUMMARY.md').read_text(encoding='utf-8')))


In [ ]:
# Optional: change to True and run this cell to download a local copy.
DOWNLOAD_RESULT_ZIP = False
if DOWNLOAD_RESULT_ZIP:
    from google.colab import files
    files.download(str(result_zip))
else:
    print('Result remains safely stored in Google Drive:', result_zip)
